# Chapitre 6 - Exercices 6.7 à 6.12

Validation croisée des polynômes locaux, densité jointe et Nadaraya-Watson, Bayes naïf
contre modèle additif logistique, critère $C_p$, dégénérescence de la vraisemblance des
mélanges gaussiens, analyse discriminante linéaire locale sur les chiffres manuscrits.

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch06_noyaux as nk
set_style()
GREEN, YELLOW, RED = COLORS[2], COLORS[3], COLORS[7]
rng = np.random.RandomState(0)

## Exercice 6.7 : Validation croisée leave-one-out des polynômes locaux

**Énoncé.** Établir une expression de la somme des carrés des résidus de validation
croisée « leave-one-out » pour la régression polynomiale locale.

**Notations.** $\hat f(x_0) = \sum_j l_j(x_0)y_j$ est l'ajustement polynomial local (degré
$k$, largeur **métrique** $\lambda$) sur toutes les données, et $\hat f^{(-i)}(x_i)$
l'ajustement en $x_i$ obtenu **sans** l'observation $i$. On note
$S_{ii} = l_i(x_i)$ le $i$-ème terme diagonal de la matrice de lissage. On cherche
$$
\mathrm{CV} = \sum_{i=1}^N\big(y_i - \hat f^{(-i)}(x_i)\big)^2 .
$$

**Étape 1 : retirer $i$ = mettre son poids à zéro.** En la cible $x_i$, l'ajustement est
un problème de moindres carrés pondérés de poids $w_j = K_\lambda(x_i, x_j)$. Avec une
largeur métrique, retirer l'observation $i$ ne modifie pas les poids des autres
observations : $\hat f^{(-i)}(x_i)$ est la valeur en $x_i$ du polynôme $\hat P^{(-i)}$ qui
minimise $\sum_{j\ne i}w_j(y_j - P(x_j))^2$.

**Étape 2 : l'astuce de la réponse modifiée.** Remplaçons $y_i$ par
$y_i^* = \hat P^{(-i)}(x_i) = \hat f^{(-i)}(x_i)$ et gardons les autres $y_j$. Pour tout
polynôme $P$ :
$$
\sum_{j\ne i}w_j(y_j - P(x_j))^2 + w_i(y_i^* - P(x_i))^2 \ \ge\ \sum_{j\ne i}w_j(y_j - \hat P^{(-i)}(x_j))^2 + 0,
$$
et l'égalité a lieu pour $P = \hat P^{(-i)}$. Donc $\hat P^{(-i)}$ est aussi l'ajustement
**avec toutes les données** et la réponse $y^*$. Par linéarité :
$$
\hat f^{(-i)}(x_i) = \sum_{j\ne i}l_j(x_i)\,y_j + l_i(x_i)\,\hat f^{(-i)}(x_i)
= \hat f(x_i) - S_{ii}y_i + S_{ii}\hat f^{(-i)}(x_i).
$$

**Étape 3 : on résout.** Si $S_{ii}\ne 1$ :
$\hat f^{(-i)}(x_i) = \dfrac{\hat f(x_i) - S_{ii}y_i}{1 - S_{ii}}$, et donc
$$
y_i - \hat f^{(-i)}(x_i) = \frac{y_i - S_{ii}y_i - \hat f(x_i) + S_{ii}y_i}{1 - S_{ii}}
= \frac{y_i - \hat f(x_i)}{1 - S_{ii}},
\qquad
\boxed{\ \mathrm{CV} = \sum_{i=1}^N\left(\frac{y_i - \hat f(x_i)}{1 - S_{ii}}\right)^2\ }
$$
Un seul ajustement suffit : il faut les résidus ordinaires et la diagonale de $\mathbf S_\lambda$.

> **Remarque.** Avec une largeur des **k plus proches voisins**, retirer $x_i$ change la
> fenêtre en $x_i$ (il faut aller chercher un voisin de plus) : les poids des autres
> observations changent et la formule n'est plus qu'une **approximation**.

Vérification numérique : formule contre $N$ réajustements, pour une largeur métrique
(égalité exacte) et une largeur de k-ppv (approximation).

In [2]:
N = 100
x = np.sort(rng.uniform(0, 1, N))
y = np.sin(4 * x) + np.sqrt(1 / 3) * rng.normal(size=N)

def brute_cv(lam=None, k=None, degree=1):
    res = []
    for i in range(N):
        m = np.arange(N) != i
        kk = None if k is None else k
        res.append(y[i] - nk.local_regression(x[i:i + 1], x[m], y[m], lam=lam, k=kk,
                                              degree=degree, kernel="tricube")[0])
    return np.sum(np.array(res) ** 2)

for deg in (0, 1, 2):
    S = nk.smoother_matrix(x, lam=0.3, degree=deg, kernel="tricube")
    print("métrique, degré %d : formule = %.6f   réajustements = %.6f"
          % (deg, N * nk.loocv_error(S, y), brute_cv(lam=0.3, degree=deg)))
S = nk.smoother_matrix(x, k=30, degree=1, kernel="tricube")
print("30-ppv,   degré 1 : formule = %.6f   réajustements = %.6f"
      % (N * nk.loocv_error(S, y), brute_cv(k=30, degree=1)))

métrique, degré 0 : formule = 36.589829   réajustements = 36.589829
métrique, degré 1 : formule = 34.878172   réajustements = 34.878172
métrique, degré 2 : formule = 35.485491   réajustements = 35.485491
30-ppv,   degré 1 : formule = 35.777138   réajustements = 35.677039
